In [5]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import json
from tqdm import tqdm
import os
import umap
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.spatial.distance import cdist, pdist
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
import re
import gc
import time
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

class DataCalculator:
    def __init__(self, config):
        self.config = config
        self.device = config.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load prompts data
        self._load_prompts_data()
        
        print("Data Calculator initialized successfully!")

    def _load_prompts_data(self):
        """Load prompts from JSONL file"""
        print(f"\nLoading prompts from {self.config['data_path']}...")
        self.prompts_data = []
        
        try:
            with open(self.config['data_path'], 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line.strip())
                    self.prompts_data.append(data)
            
            self.prompts = [data['text'] for data in self.prompts_data]
            self.labels = [data['role'] for data in self.prompts_data]
            self.prompt_ids = [data['id'] for data in self.prompts_data]
            
            print(f"Loaded {len(self.prompts)} prompts")
            print(f"Categories: {set(self.labels)}")
            
        except Exception as e:
            print(f"Error loading prompts: {e}")
            raise e

    def _cleanup_memory(self):
        """Clean up GPU memory"""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        gc.collect()

    def setup_model(self, model_path):
        """Initialize model"""
        self._cleanup_memory()
        
        print(f"\nLoading model from {model_path}...")
        try:
            start_time = time.time()
            
            tokenizer = AutoTokenizer.from_pretrained(
                model_path,
                trust_remote_code=True
            )
            
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model.eval()
            
            load_time = time.time() - start_time
            print(f"Model loaded successfully! Time: {load_time:.2f}s")
            
            return tokenizer, model
        except Exception as e:
            print(f"Error loading model {model_path}: {e}")
            raise e

    def extract_representations(self, model_path):
        """Extract representations using mean pooling"""
        print(f"\nExtracting representations: {model_path}")
        
        tokenizer, model = self.setup_model(model_path)
        representations = []
        
        print(f"Extracting representations ({len(self.prompts)} samples)...")
        
        batch_size = 8
        num_batches = (len(self.prompts) + batch_size - 1) // batch_size
        
        with torch.no_grad():
            for batch_idx in tqdm(range(num_batches), desc="Extracting"):
                start_idx = batch_idx * batch_size
                end_idx = min((batch_idx + 1) * batch_size, len(self.prompts))
                
                batch_prompts = self.prompts[start_idx:end_idx]
                
                try:
                    inputs = tokenizer(
                        batch_prompts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.config['max_length'],
                        return_attention_mask=True
                    ).to(self.device)
                    
                    outputs = model(**inputs, output_hidden_states=True)
                    hidden_states = outputs.hidden_states[-1]
                    attention_mask = inputs['attention_mask']
                    
                    for i in range(len(batch_prompts)):
                        sample_hidden = hidden_states[i:i+1]
                        sample_mask = attention_mask[i:i+1]
                        
                        input_mask_expanded = sample_mask.unsqueeze(-1).expand(sample_hidden.size()).float()
                        sum_embeddings = torch.sum(sample_hidden * input_mask_expanded, 1)
                        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                        mean_rep = sum_embeddings / sum_mask
                        
                        representations.append(mean_rep.squeeze().cpu().numpy())
                
                except Exception as e:
                    print(f"Error processing batch {batch_idx}: {e}")
                    dummy_shape = 2560
                    representations.extend([np.zeros(dummy_shape)] * len(batch_prompts))
        
        del model, tokenizer
        self._cleanup_memory()
        
        X = np.array(representations)
        print(f"Representation extraction completed, shape: {X.shape}")
        
        return X

    def compute_projections(self, X):
        """Compute dimensionality reduction projections"""
        print("\nComputing projections...")
        
        projections = {}
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # UMAP
        print("  Computing UMAP...")
        try:
            reducer = umap.UMAP(
                n_components=2,
                n_neighbors=min(20, X_scaled.shape[0] // 4),
                min_dist=0.1,
                metric='cosine',
                random_state=42,
                n_jobs=1
            )
            projections['umap'] = reducer.fit_transform(X_scaled)
            print("  UMAP completed")
        except Exception as e:
            print(f"  UMAP failed: {e}")
            projections['umap'] = X_scaled[:, :2]
        
        # PCA
        print("  Computing PCA...")
        try:
            pca = PCA(n_components=2, random_state=42)
            projections['pca'] = pca.fit_transform(X_scaled)
            print("  PCA completed")
        except Exception as e:
            print(f"  PCA failed: {e}")
            projections['pca'] = projections['umap'].copy()
        
        return projections

    def calculate_detailed_metrics(self, X, projections):
        """Calculate detailed metrics"""
        print("\nCalculating detailed metrics...")
        
        metrics = {}
        
        # Basic metrics
        for proj_name, X_proj in projections.items():
            df = pd.DataFrame({
                'D1': X_proj[:, 0],
                'D2': X_proj[:, 1],
                'Category': self.labels,
                'ID': self.prompt_ids
            })
            
            # 1. Global clustering quality
            category_labels = pd.factorize(df['Category'])[0]
            
            if len(set(category_labels)) > 1:
                try:
                    metrics[f'{proj_name}_silhouette'] = silhouette_score(X_proj[:, :2], category_labels)
                    metrics[f'{proj_name}_calinski_harabasz'] = calinski_harabasz_score(X_proj[:, :2], category_labels)
                    metrics[f'{proj_name}_davies_bouldin'] = davies_bouldin_score(X_proj[:, :2], category_labels)
                except:
                    pass
            
            # 2. Separation analysis
            self_mask = df['Category'] == 'Self-Prompts'
            baseline_mask = df['Category'] == 'Baseline-Prompts'
            
            if self_mask.any() and baseline_mask.any():
                self_points = df.loc[self_mask, ['D1', 'D2']].values
                baseline_points = df.loc[baseline_mask, ['D1', 'D2']].values
                
                if len(self_points) > 0 and len(baseline_points) > 0:
                    # Inter-class distance
                    inter_dist = cdist(self_points, baseline_points).mean()
                    metrics[f'{proj_name}_inter_distance'] = inter_dist
                    
                    # Intra-class distances
                    if len(self_points) > 1:
                        intra_self = pdist(self_points).mean()
                        metrics[f'{proj_name}_intra_self'] = intra_self
                    
                    if len(baseline_points) > 1:
                        intra_baseline = pdist(baseline_points).mean()
                        metrics[f'{proj_name}_intra_baseline'] = intra_baseline
        
        # 3. High-dimensional metrics
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # High-dimensional silhouette
        category_labels = pd.factorize(self.labels)[0]
        if len(set(category_labels)) > 1:
            try:
                # Use PCA for high-dim silhouette to reduce computation
                pca_50 = PCA(n_components=min(50, X_scaled.shape[1]), random_state=42)
                X_pca = pca_50.fit_transform(X_scaled)
                metrics['high_dim_silhouette'] = silhouette_score(X_pca[:, :10], category_labels)
            except:
                pass
        
        return metrics

    def analyze_self_dimensions(self, projections):
        """Analyze self-consciousness dimensions"""
        print("\nAnalyzing self-consciousness dimensions...")
        
        dimension_analysis = {}
        
        for proj_name, X_proj in projections.items():
            df = pd.DataFrame({
                'D1': X_proj[:, 0],
                'D2': X_proj[:, 1],
                'Category': self.labels,
                'ID': self.prompt_ids
            })
            
            df_self = df[df['Category'] == 'Self-Prompts']
            if len(df_self) == 0:
                continue
            
            # Extract dimension information
            def extract_dimension(pid):
                match = re.search(r'(\d+)', str(pid))
                return int(match.group(1)) if match else 0
            
            df_self['Dimension'] = df_self['ID'].apply(extract_dimension)
            
            dim_metrics = {}
            for dim in range(1, 16):
                dim_points = df_self[df_self['Dimension'] == dim][['D1', 'D2']].values
                
                if len(dim_points) > 5:
                    # Compactness
                    compactness = pdist(dim_points).mean() if len(dim_points) > 1 else 0
                    
                    # Separation from other dimensions
                    other_dims = df_self[df_self['Dimension'] != dim][['D1', 'D2']].values
                    if len(other_dims) > 0:
                        separation = cdist(dim_points, other_dims).mean()
                        
                        dim_metrics[dim] = {
                            'n_samples': len(dim_points),
                            'compactness': compactness,
                            'separation': separation,
                            'separation_ratio': separation / max(compactness, 0.001)
                        }
            
            dimension_analysis[proj_name] = dim_metrics
        
        return dimension_analysis

    def calculate_and_save_all_data(self):
        """Calculate all data and save to Excel"""
        print("="*70)
        print("Calculating All Data for Training Comparison")
        print("="*70)
        
        # Create Excel writer
        excel_path = 'umap_analysis_data.xlsx'
        print(f"\nResults will be saved to: {excel_path}")
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            all_data = {}
            
            # 1. Analyze pre-trained model
            print("\n" + "="*60)
            print("Analyzing Pre-trained Model")
            print("="*60)
            
            before_start = time.time()
            X_before = self.extract_representations(self.config['base_model'])
            projections_before = self.compute_projections(X_before)
            metrics_before = self.calculate_detailed_metrics(X_before, projections_before)
            dim_analysis_before = self.analyze_self_dimensions(projections_before)
            before_time = time.time() - before_start
            
            # Save before data
            before_data = {
                'X': X_before,
                'projections': projections_before,
                'metrics': metrics_before,
                'dimension_analysis': dim_analysis_before
            }
            
            # Create dataframes for before model
            df_before_umap = pd.DataFrame(projections_before['umap'], columns=['UMAP1', 'UMAP2'])
            df_before_umap['Category'] = self.labels
            df_before_umap['ID'] = self.prompt_ids
            df_before_umap['Model'] = 'Pre-trained'
            
            df_before_pca = pd.DataFrame(projections_before['pca'], columns=['PCA1', 'PCA2'])
            df_before_pca['Category'] = self.labels
            df_before_pca['ID'] = self.prompt_ids
            df_before_pca['Model'] = 'Pre-trained'
            
            # Metrics dataframe
            df_metrics_before = pd.DataFrame([metrics_before])
            df_metrics_before['Model'] = 'Pre-trained'
            
            # Save to Excel
            df_before_umap.to_excel(writer, sheet_name='Before_UMAP', index=False)
            df_before_pca.to_excel(writer, sheet_name='Before_PCA', index=False)
            df_metrics_before.to_excel(writer, sheet_name='Before_Metrics', index=False)
            
            print(f"Pre-trained model analysis completed, time: {before_time:.2f}s")
            
            # Cleanup
            print("\nCleaning up memory...")
            self._cleanup_memory()
            time.sleep(2)
            
            # 2. Analyze fine-tuned model
            print("\n" + "="*60)
            print("Analyzing Fine-tuned Model")
            print("="*60)
            
            after_start = time.time()
            X_after = self.extract_representations(self.config['trained_model'])
            projections_after = self.compute_projections(X_after)
            metrics_after = self.calculate_detailed_metrics(X_after, projections_after)
            dim_analysis_after = self.analyze_self_dimensions(projections_after)
            after_time = time.time() - after_start
            
            # Save after data
            after_data = {
                'X': X_after,
                'projections': projections_after,
                'metrics': metrics_after,
                'dimension_analysis': dim_analysis_after
            }
            
            # Create dataframes for after model
            df_after_umap = pd.DataFrame(projections_after['umap'], columns=['UMAP1', 'UMAP2'])
            df_after_umap['Category'] = self.labels
            df_after_umap['ID'] = self.prompt_ids
            df_after_umap['Model'] = 'Fine-tuned'
            
            df_after_pca = pd.DataFrame(projections_after['pca'], columns=['PCA1', 'PCA2'])
            df_after_pca['Category'] = self.labels
            df_after_pca['ID'] = self.prompt_ids
            df_after_pca['Model'] = 'Fine-tuned'
            
            # Metrics dataframe
            df_metrics_after = pd.DataFrame([metrics_after])
            df_metrics_after['Model'] = 'Fine-tuned'
            
            # Save to Excel
            df_after_umap.to_excel(writer, sheet_name='After_UMAP', index=False)
            df_after_pca.to_excel(writer, sheet_name='After_PCA', index=False)
            df_metrics_after.to_excel(writer, sheet_name='After_Metrics', index=False)
            
            print(f"Fine-tuned model analysis completed, time: {after_time:.2f}s")
            
            # 3. Create comparison dataframe
            print("\n" + "="*60)
            print("Creating Comparison Data")
            print("="*60)
            
            # Combine UMAP data
            df_umap_combined = pd.concat([df_before_umap, df_after_umap], ignore_index=True)
            df_pca_combined = pd.concat([df_before_pca, df_after_pca], ignore_index=True)
            
            # Combine metrics
            df_metrics_combined = pd.concat([df_metrics_before, df_metrics_after], ignore_index=True)
            
            # Key metrics comparison
            key_metrics = [
                ('Silhouette Score (UMAP)', 'umap_silhouette'),
                ('Calinski-Harabasz (UMAP)', 'umap_calinski_harabasz'),
                ('Davies-Bouldin (UMAP)', 'umap_davies_bouldin'),
                ('Inter-class Distance (UMAP)', 'umap_inter_distance'),
                ('Intra-Self Distance (UMAP)', 'umap_intra_self'),
                ('Intra-Baseline Distance (UMAP)', 'umap_intra_baseline'),
                ('Silhouette Score (PCA)', 'pca_silhouette'),
                ('Calinski-Harabasz (PCA)', 'pca_calinski_harabasz')
            ]
            
            comparison_data = []
            for display_name, metric_key in key_metrics:
                if metric_key in metrics_before and metric_key in metrics_after:
                    before_val = metrics_before[metric_key]
                    after_val = metrics_after[metric_key]
                    change = after_val - before_val
                    change_pct = (change / abs(before_val) * 100) if before_val != 0 else 0
                    
                    comparison_data.append({
                        'Metric': display_name,
                        'Pre-trained': before_val,
                        'Fine-tuned': after_val,
                        'Absolute Change': change,
                        'Percentage Change': change_pct
                    })
            
            df_comparison = pd.DataFrame(comparison_data)
            
            # Dimension analysis comparison
            dim_comparison_data = []
            if 'umap' in dim_analysis_before and 'umap' in dim_analysis_after:
                for dim in range(1, 16):
                    if dim in dim_analysis_before['umap'] and dim in dim_analysis_after['umap']:
                        before_ratio = dim_analysis_before['umap'][dim]['separation_ratio']
                        after_ratio = dim_analysis_after['umap'][dim]['separation_ratio']
                        change = after_ratio - before_ratio
                        
                        dim_comparison_data.append({
                            'Dimension': dim,
                            'Pre-trained Separation Ratio': before_ratio,
                            'Fine-tuned Separation Ratio': after_ratio,
                            'Separation Ratio Change': change
                        })
            
            if dim_comparison_data:
                df_dim_comparison = pd.DataFrame(dim_comparison_data)
                df_dim_comparison.to_excel(writer, sheet_name='Dimension_Comparison', index=False)
            
            # Save comparison sheets
            df_umap_combined.to_excel(writer, sheet_name='Combined_UMAP', index=False)
            df_pca_combined.to_excel(writer, sheet_name='Combined_PCA', index=False)
            df_metrics_combined.to_excel(writer, sheet_name='Combined_Metrics', index=False)
            df_comparison.to_excel(writer, sheet_name='Key_Metrics_Comparison', index=False)
            
            # Create summary statistics
            summary_data = {
                'Analysis Phase': ['Pre-trained Model', 'Fine-tuned Model'],
                'Analysis Time (s)': [before_time, after_time],
                'Total Samples': [len(self.prompts), len(self.prompts)],
                'Self-Prompts': [self.labels.count('Self-Prompts'), self.labels.count('Self-Prompts')],
                'Baseline-Prompts': [self.labels.count('Baseline-Prompts'), self.labels.count('Baseline-Prompts')]
            }
            
            df_summary = pd.DataFrame(summary_data)
            df_summary.to_excel(writer, sheet_name='Summary', index=False)
            
            print(f"\nAll data has been saved to {excel_path}")
            print(f"Total sheets created:")
            print("  1. Before_UMAP, Before_PCA, Before_Metrics")
            print("  2. After_UMAP, After_PCA, After_Metrics")
            print("  3. Combined_UMAP, Combined_PCA, Combined_Metrics")
            print("  4. Key_Metrics_Comparison, Dimension_Comparison, Summary")
            
            return all_data

# Configuration
config = {
    'base_model': 'unsloth/Qwen3-4B-Instruct-2507',
    'trained_model': './best_model_gen_20',
    'data_path': 'sb.jsonl',
    'max_length': 512,
    'output_dir': 'analysis_results',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Run the analysis
if __name__ == "__main__":
    if not os.path.exists(config['data_path']):
        print(f"Error: Data file {config['data_path']} not found!")
    else:
        print("Starting comprehensive data calculation...")
        print("This will calculate all metrics and save to umap_analysis_data.xlsx")
        print("\nNote: This may take several minutes depending on your hardware.")
        
        calculator = DataCalculator(config)
        calculator.calculate_and_save_all_data()
        
        print("\n" + "="*70)
        print("Data calculation completed successfully!")
        print("All results saved to 'umap_analysis_data.xlsx'")
        print("="*70)

Starting comprehensive data calculation...
This will calculate all metrics and save to umap_analysis_data.xlsx

Note: This may take several minutes depending on your hardware.

Loading prompts from sb.jsonl...
Loaded 480 prompts
Categories: {'Baseline-Prompts', 'Self-Prompts'}
Data Calculator initialized successfully!
Calculating All Data for Training Comparison

Results will be saved to: umap_analysis_data.xlsx

Analyzing Pre-trained Model

Extracting representations: unsloth/Qwen3-4B-Instruct-2507

Loading model from unsloth/Qwen3-4B-Instruct-2507...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully! Time: 1.91s
Extracting representations (480 samples)...


Extracting: 100%|██████████| 60/60 [00:01<00:00, 39.72it/s]


Representation extraction completed, shape: (480, 2560)

Computing projections...
  Computing UMAP...
  UMAP completed
  Computing PCA...
  PCA completed

Calculating detailed metrics...

Analyzing self-consciousness dimensions...
Pre-trained model analysis completed, time: 5.13s

Cleaning up memory...

Analyzing Fine-tuned Model

Extracting representations: ./best_model_gen_20

Loading model from ./best_model_gen_20...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully! Time: 1.84s
Extracting representations (480 samples)...


Extracting: 100%|██████████| 60/60 [00:01<00:00, 38.66it/s]


Representation extraction completed, shape: (480, 2560)

Computing projections...
  Computing UMAP...
  UMAP completed
  Computing PCA...
  PCA completed

Calculating detailed metrics...

Analyzing self-consciousness dimensions...
Fine-tuned model analysis completed, time: 5.13s

Creating Comparison Data

All data has been saved to umap_analysis_data.xlsx
Total sheets created:
  1. Before_UMAP, Before_PCA, Before_Metrics
  2. After_UMAP, After_PCA, After_Metrics
  3. Combined_UMAP, Combined_PCA, Combined_Metrics
  4. Key_Metrics_Comparison, Dimension_Comparison, Summary

Data calculation completed successfully!
All results saved to 'umap_analysis_data.xlsx'
